# Hadoop Local Cluster — Hands-on Guide

Interactive walkthrough of the Docker-based Hadoop cluster in `hadoop-local-docker/`.

Uses a fictional e-commerce company **ShopStream** (orders, reviews, clickstream) to show how each Hadoop service works in practice.

**Prerequisite:** Cluster running — complete [HADOOP-STUDENT-GUIDE.md](./HADOOP-STUDENT-GUIDE.md) Steps 1–4 first.

**How to use this guide:** Run cells in order. Markdown cells explain concepts; code cells run commands against your local cluster.

| Web UI | URL |
|--------|-----|
| NameNode (HDFS) | http://localhost:9870 |
| YARN ResourceManager | http://localhost:8088 |
| MapReduce History | http://localhost:19888 |

## 1. Why Hadoop for e-commerce

**ShopStream** generates three types of data every day:

| Data | Sample file | Real-world scale |
|------|-------------|------------------|
| Orders | `data/ecommerce/orders.csv` | Millions of rows/day |
| Product reviews | `data/ecommerce/product_reviews.txt` | Terabytes over time |
| Clickstream | `data/ecommerce/clickstream.csv` | Billions of events/day |

ShopStream needs to:
1. **Store** large files reliably across multiple machines
2. **Process** batches overnight (top products, trends, fraud checks)
3. **Schedule** compute jobs across a cluster

| Hadoop component | Role | E-commerce analogy |
|------------------|------|-------------------|
| **HDFS** | Distributed storage | Warehouse shelves across multiple racks |
| **YARN** | Resource scheduler | Shift manager assigning work to teams |
| **MapReduce** | Batch processing | Assembly line — split work, combine results |

## 2. Cluster architecture

Seven Docker containers simulate a mini data center:

| Container | Hadoop role | ShopStream analogy | Port |
|-----------|-------------|-------------------|------|
| `namenode` | HDFS NameNode | Inventory catalog — tracks where file blocks live | 9870, 9900 |
| `worker-1/2/3` | DataNode | Warehouse racks — store replicated file blocks | 18042, 28042, 38042 |
| `worker-1/2/3` | NodeManager | Analyst desks — CPU/RAM for batch jobs | (same workers) |
| `resourcemanager` | YARN scheduler | Operations manager — assigns jobs to workers | 8088 |
| `historyserver` | Job history | Audit log of completed batch jobs | 19888 |
| `proxyserver` | YARN proxy | Gateway to running application UIs | 9099 |

## 3. Verify the cluster

Confirm all services are running before working with HDFS or MapReduce.

In [ ]:
# List all containers and their status
!docker compose ps

In [ ]:
# Check health of each service (all should report "healthy")
!docker inspect --format='{{.Name}} {{.State.Health.Status}}' namenode resourcemanager historyserver proxyserver worker-1 worker-2 worker-3

In [ ]:
# Confirm YARN registered 3 worker nodes
!docker exec resourcemanager yarn node -list

**Expected:** 7 containers healthy, 3 YARN nodes in `RUNNING` state.

Open http://localhost:9870 → **Datanodes** → **Live Nodes: 3**

## 4. HDFS — store ShopStream data

**HDFS** (Hadoop Distributed File System) splits files into blocks and replicates them across DataNodes.

The **NameNode** holds metadata (file paths, block locations). **DataNodes** on `worker-1/2/3` store the actual bytes.

Example: a 10 GB orders file becomes ~80 blocks, each copied to 2 racks (replication factor 2 in our cluster).

In [ ]:
# Create HDFS directory layout (like S3 prefixes in a data lake)
!docker exec namenode hdfs dfs -mkdir -p /shopstream/raw/orders
!docker exec namenode hdfs dfs -mkdir -p /shopstream/raw/reviews
!docker exec namenode hdfs dfs -mkdir -p /shopstream/raw/clickstream
!docker exec namenode hdfs dfs -mkdir -p /shopstream/processed

!docker exec namenode hdfs dfs -ls -R /shopstream

In [ ]:
# Copy local e-commerce files into the namenode container, then upload to HDFS
!docker cp data/ecommerce/orders.csv namenode:/tmp/orders.csv
!docker cp data/ecommerce/product_reviews.txt namenode:/tmp/product_reviews.txt
!docker cp data/ecommerce/clickstream.csv namenode:/tmp/clickstream.csv

!docker exec namenode hdfs dfs -put -f /tmp/orders.csv /shopstream/raw/orders/orders.csv
!docker exec namenode hdfs dfs -put -f /tmp/product_reviews.txt /shopstream/raw/reviews/product_reviews.txt
!docker exec namenode hdfs dfs -put -f /tmp/clickstream.csv /shopstream/raw/clickstream/clickstream.csv

print("Uploaded orders, reviews, and clickstream to HDFS")

In [ ]:
# List and preview orders stored in HDFS
!docker exec namenode hdfs dfs -ls /shopstream/raw/orders
!echo "--- First 5 rows ---"
!docker exec namenode hdfs dfs -cat /shopstream/raw/orders/orders.csv | head -6

In [ ]:
# Show file size and replication factor
!docker exec namenode hdfs dfs -stat "%n | size=%b bytes | replication=%r" /shopstream/raw/orders/orders.csv

In the NameNode UI (http://localhost:9870) → **Utilities → Browse the file system** → navigate to `/shopstream/raw/orders/orders.csv` to see which workers hold each block.

Production e-commerce systems often use replication factor **3** across availability zones.

## 5. YARN — schedule batch jobs

**YARN** (Yet Another Resource Negotiator) decides which worker runs each job and how much CPU/memory it gets.

ShopStream nightly batch examples:
- Calculate top 10 products by revenue
- Flag fraudulent orders
- Aggregate clickstream conversion funnels

| YARN term | Meaning |
|-----------|--------|
| ResourceManager | Cluster scheduler (`resourcemanager` container) |
| NodeManager | Per-worker agent (`worker-1/2/3`) |
| Application | One batch job (e.g. wordcount report) |
| Container | CPU/RAM slice assigned to a task |

In [ ]:
!docker exec resourcemanager yarn node -list -all

Open http://localhost:8088 → **Nodes** to see worker capacity in the UI.

## 6. MapReduce — analyze product reviews

**Business question:** *What words appear most often in ShopStream product reviews?*

MapReduce process:
1. **Map** — read review lines, emit `(word, 1)` for each word
2. **Shuffle** — group identical words together
3. **Reduce** — sum counts → `delivery\t3`, `excellent\t2`, ...

In [ ]:
# Remove previous output so the job can be re-run
!docker exec namenode hdfs dfs -rm -r -f /shopstream/processed/review_wordcount 2>/dev/null || true

# Submit wordcount job to YARN
!docker exec namenode bash -lc \
  'hadoop jar $(ls $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar | head -1) wordcount /shopstream/raw/reviews/product_reviews.txt /shopstream/processed/review_wordcount'

Track the job at http://localhost:8088/cluster/apps — status moves from `ACCEPTED` → `RUNNING` → `FINISHED`.

In [ ]:
# Top word counts from review text
!docker exec namenode hdfs dfs -cat /shopstream/processed/review_wordcount/part-r-00000 | sort -t$'\t' -k2 -nr | head -15

### Filter reviews mentioning delivery (grep example)

MapReduce `grep` filters lines matching a pattern — similar to searching logs at scale.

In [ ]:
!docker exec namenode hdfs dfs -rm -r -f /shopstream/processed/delivery_reviews 2>/dev/null || true

!docker exec namenode bash -lc \
  'hadoop jar $(ls $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar | head -1) grep /shopstream/raw/reviews/product_reviews.txt /shopstream/processed/delivery_reviews "delivery.*"'

!docker exec namenode hdfs dfs -cat /shopstream/processed/delivery_reviews/part-m-00000

## 7. History Server and Proxy Server

| Service | Purpose | URL |
|---------|---------|-----|
| **historyserver** | Logs and metrics for finished MapReduce jobs | http://localhost:19888 |
| **proxyserver** | Routes browser traffic to running YARN applications | http://localhost:9099 |

Use the History Server to inspect job duration, counters, and failures — similar to an audit trail in production pipelines.

In [ ]:
!docker exec resourcemanager yarn application -list -appStates FINISHED

## 8. End-to-end batch pipeline

How ShopStream fits together:

```
  App / Website                    Nightly batch (Hadoop)
  ─────────────                    ──────────────────────
  Orders, reviews, events  ──►   HDFS /shopstream/raw/
                                        │
                                        ▼
                                   MapReduce / Spark (YARN)
                                        │
                                        ▼
                                   HDFS /shopstream/processed/
                                        │
                                        ▼
                                   BI dashboards, ML features
```

### Map local cluster → AWS

| Local | AWS equivalent |
|-------|----------------|
| HDFS | Amazon S3 |
| YARN + MapReduce | Amazon EMR |
| NameNode catalog | S3 + AWS Glue Catalog |
| Batch jobs | EMR steps, Glue jobs, Lambda triggers |

## Reference

| Task | Command |
|------|---------|
| Start cluster | `docker pull neshkeev/hadoop:3.3.6-jdk-11` then `docker compose up -d` |
| Check status | `docker compose ps` |
| Stop cluster | `docker compose down` |
| Full setup details | [HADOOP-STUDENT-GUIDE.md](./HADOOP-STUDENT-GUIDE.md) |

**Sample data:** `data/ecommerce/orders.csv`, `product_reviews.txt`, `clickstream.csv`